In [24]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)

DATA_DIR = Path("raw_data/")          # adjust to your layout
INFILE_NODES = DATA_DIR / "new_nodes_v3.xlsx"
INFILE_EDGES = DATA_DIR / "old_edges.xlsx"
OUT_DIR = Path("out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [25]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [26]:
df_nodes = pd.read_excel(
    INFILE_NODES,
    sheet_name="master",
    engine="openpyxl"
)

print("Shape:", df_nodes.shape)
df_nodes.head()

Shape: (296, 18)


,node_id,name,node_type,org_type,governance_level,geographic_scale,functional_domain,roles,fema_lifeline,url,key_contact,contact_url,Link to resources,summary,review_flag,review_note,AER Questions (verify home pg & func domain),Unnamed: 17
0,211info,211info,organization,nonprofit_community,non_governmental,Oregon,community_resilience,"coordination, data_tools_provider",communications,https://www.211info.org/,NaN,NaN,NaN,Provides community information and referral services that support disaster preparedness and recovery.,NaN,NaN,NaN,NaN
1,AF8,Alpine Fault Magnitude 8,hub,coordination_structure,multi,International,community_resilience,"coordination, knowledge_provider, data_tools_provider",NaN,https://af8.org.nz/,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AirNG,Air National Guard,organization,government,state,U.S. National,emergency_management,infrastructure_operator,safety_security,https://www.ang.af.mil/,NaN,NaN,NaN,"Provides airlift, logistics, and emergency support capabilities for disaster response.",yes,Generic umbrella entry; consider replacing with state-specific Air National Guard entities.,NaN,NaN
3,AlertFM,Global Security Systems/ALERT FM,organization,private_sector,private,International,emergency_management,"messaging_alerts_provider, data_tools_provider",communications,https://alertfm.com/,NaN,NaN,NaN,Provides alerting technology and communications tools used for emergency warning and public information.,NaN,NaN,NaN,NaN
4,AlertReady,Alert Ready Emergency Alert System,program,coordination_structure,multi,Canada,emergency_management,messaging_alerts_provider,communications,https://www.alertready.ca/,NaN,NaN,NaN,Delivers public emergency alerts across Canada through participating broadcasters and wireless providers.,yes,Could be modeled as a national alerting system rather than a standalone program node.,NaN,NaN


## node id cleaning

In [27]:
df = df_nodes.copy()

missing_node_id = df["node_id"].isna() | df["node_id"].astype(str).str.strip().eq("")
dropped_rows = df.loc[missing_node_id].copy()

print("Number of rows that will be dropped:", len(dropped_rows))
print()
print("Names being dropped:")
display(dropped_rows["name"].fillna("(missing name)").value_counts(dropna=False))

df = df.loc[~missing_node_id].copy()

print()
print("Shape after dropping missing node_id:", df.shape)

Number of rows that will be dropped: 12

Names being dropped:


Spirit Lake-Toutle-Cowlitz River Collaborative                                           1
Cowlitz County PUD                                                                       1
DEPARTMENT OF METEOROLOGY AND CLIMATE SCIENCE, FEDERAL UNIVERSITY OF TECHNOLOGY AKURE    1
Gombe State University                                                                   1
Indonesia Agency for Meteorology Climatology and Geophysics (BMKG)                       1
New Mexico State University                                                              1
Pakistan Meteorological Department                                                       1
Penn State University                                                                    1
Southern Methodist University                                                            1
University of Edinburgh                                                                  1
University of Houston                                                                    1


Shape after dropping missing node_id: (284, 18)


In [28]:
df["node_id"] = (
    df["node_id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
)

dup_node_id = df[df.duplicated("node_id", keep=False)].sort_values("node_id")

print("Number of rows with duplicated node_id:", len(dup_node_id))
display(dup_node_id)

Number of rows with duplicated node_id: 0


,node_id,name,node_type,org_type,governance_level,geographic_scale,functional_domain,roles,fema_lifeline,url,key_contact,contact_url,Link to resources,summary,review_flag,review_note,AER Questions (verify home pg & func domain),Unnamed: 17


In [29]:
MISSING_TOKENS = {"", "nan", "none", "n/a", "na", "null", "???"}

def normalize_text(value):
    if pd.isna(value):
        return ""
    s = str(value).strip()
    s = re.sub(r"\s+", " ", s)
    return "" if s.lower() in MISSING_TOKENS else s

def pretty_label(token):
    token = normalize_text(token)
    if not token:
        return "Other"
    if "_" in token:
        return token.replace("_", " ").title()
    if token.islower():
        return token.title()
    return token

def split_categories(value):
    s = normalize_text(value)
    if not s:
        return ["Other"]

    parts = re.split(r"\s*[,;/|]\s*", s)
    out = []
    for part in parts:
        label = pretty_label(part)
        if label not in out:
            out.append(label)
    return out or ["Other"]

display(df[["name", "node_id"]].head(20))

,name,node_id
0,211info,211info
1,Alpine Fault Magnitude 8,AF8
2,Air National Guard,AirNG
3,Global Security Systems/ALERT FM,AlertFM
4,Alert Ready Emergency Alert System,AlertReady
5,Aon Impact Forecasting,AON
6,American Society of Civil Engineers,ASCE
7,Affiliated Tribes of Northwest Indians,ATNI
8,City of Bainbridge Island,Bainbridge
9,Bathymetrix,Bathymetrix


## categorical fields

In [30]:
CATEGORY_COLUMNS = [
    "node_type",
    "org_type",
    "governance_level",
    "geographic_scale",
    "functional_domain",
    "roles",
    "fema_lifeline",
]

for col in CATEGORY_COLUMNS:
    print(f"{col}: {df[col].nunique(dropna=True)} unique")
    display(df[col].fillna("(missing)").value_counts(dropna=False).head(20))
    print()

node_type: 4 unique


organization         195
Tribe/FirstNation     53
hub                   22
program               14
Name: node_type, dtype: int64


org_type: 8 unique


government                83
tribal_sovereign          53
coordination_structure    36
nonprofit_community       35
academic                  33
private_sector            24
quasi_governmental        14
media                      6
Name: org_type, dtype: int64


governance_level: 7 unique


...

community_resilience      85
earthquake_science        74
emergency_management      65
infrastructure_systems    60
Name: functional_domain, dtype: int64


roles: 30 unique


(missing)                                                71
coordination                                             43
knowledge_provider                                       36
infrastructure_operator                                  27
policy_maker_regulator                                   21
coordination, policy_maker_regulator                      9
coordination, knowledge_provider                          8
messaging_alerts_provider                                 8
data_tools_provider, knowledge_provider                   8
knowledge_provider, data_tools_provider                   7
emergency_response                                        6
infrastructure_operator, policy_maker_regulator           6
coordination, data_tools_provider                         5
data_tools_provider                                       4
coordination, knowledge_provider, data_tools_provider     3
coordination, messaging_alerts_provider                   2
knowledge_provider, coordination        


fema_lifeline: 9 unique


(missing)                 215
transportation             17
communications             16
energy                     11
safety_security             9
water_systems               7
health_medical              4
energy, water_systems       2
energy, communications      2
food_hydration_shelter      1
Name: fema_lifeline, dtype: int64

In [31]:
for col in CATEGORY_COLUMNS:
    df[f"{col}_clean"] = df[col].apply(normalize_text)

df[[f"{col}_clean" for col in CATEGORY_COLUMNS]].head()

,node_type_clean,org_type_clean,governance_level_clean,geographic_scale_clean,functional_domain_clean,roles_clean,fema_lifeline_clean
0,organization,nonprofit_community,non_governmental,Oregon,community_resilience,"coordination, data_tools_provider",communications
1,hub,coordination_structure,multi,International,community_resilience,"coordination, knowledge_provider, data_tools_provider",
2,organization,government,state,U.S. National,emergency_management,infrastructure_operator,safety_security
3,organization,private_sector,private,International,emergency_management,"messaging_alerts_provider, data_tools_provider",communications
4,program,coordination_structure,multi,Canada,emergency_management,messaging_alerts_provider,communications


In [32]:
CATEGORY_EXPORTS = {
    "node_type": ("nodeTypes", "nodeTypePrimary"),
    "org_type": ("orgTypes", "orgTypePrimary"),
    "governance_level": ("governanceLevels", "governanceLevelPrimary"),
    "geographic_scale": ("geoTags", "geoPrimary"),
    "functional_domain": ("functionalDomains", "functionalDomainPrimary"),
    "roles": ("roleTags", "rolePrimary"),
    "fema_lifeline": ("lifelineTags", "femaLifelinePrimary"),
}

In [33]:
for source_col, (list_col, primary_col) in CATEGORY_EXPORTS.items():
    clean_col = f"{source_col}_clean"
    df[list_col] = df[clean_col].apply(split_categories)
    df[primary_col] = df[list_col].str[0]

df[[
    "name",
    "node_id",
    "orgTypePrimary",
    "geoPrimary",
    "nodeTypePrimary",
    "governanceLevelPrimary",
    "functionalDomainPrimary",
    "rolePrimary",
    "femaLifelinePrimary",
]].head(20)

,name,node_id,orgTypePrimary,geoPrimary,nodeTypePrimary,governanceLevelPrimary,functionalDomainPrimary,rolePrimary,femaLifelinePrimary
0,211info,211info,Nonprofit Community,Oregon,Organization,Non Governmental,Community Resilience,Coordination,Communications
1,Alpine Fault Magnitude 8,AF8,Coordination Structure,International,Hub,Multi,Community Resilience,Coordination,Other
2,Air National Guard,AirNG,Government,U.S. National,Organization,State,Emergency Management,Infrastructure Operator,Safety Security
3,Global Security Systems/ALERT FM,AlertFM,Private Sector,International,Organization,Private,Emergency Management,Messaging Alerts Provider,Communications
4,Alert Ready Emergency Alert System,AlertReady,Coordination Structure,Canada,Program,Multi,Emergency Management,Messaging Alerts Provider,Communications
5,Aon Impact Forecasting,AON,Private Sector,International,Organization,Non Governmental,Infrastructure Systems,Knowledge Provider,Other
6,American Society of Civil Engineers,ASCE,Nonprofit Community,U.S. National,Organization,Non Governmental,Infrastructure Systems,Knowledge Provider,Other
7,Affiliated Tribes of Northwest Indians,ATNI,Tribal Sovereign,PNW Regional,Tribe,Non Governmental,Community Resilience,Coordination,Other
8,City of Bainbridge Island,Bainbridge,Government,Washington,Organization,Local,Emergency Management,Coordination,Other
9,Bathymetrix,Bathymetrix,Private Sector,Oregon,Organization,Non Governmental,Infrastructure Systems,Data Tools Provider,Other


In [34]:
for source_col, (list_col, primary_col) in CATEGORY_EXPORTS.items():
    labels = sorted({label for values in df[list_col] for label in values})
    print(primary_col)
    print(labels)
    print()

display(df[["orgTypePrimary", "geoPrimary"]].value_counts().head(20))

nodeTypePrimary
['FirstNation', 'Hub', 'Organization', 'Program', 'Tribe']

orgTypePrimary
['Academic', 'Coordination Structure', 'Government', 'Media', 'Nonprofit Community', 'Private Sector', 'Quasi Governmental', 'Tribal Sovereign']

governanceLevelPrimary
['Federal', 'Local', 'Multi', 'Non Governmental', 'Private', 'Sovereign', 'State']

geoPrimary
['British Columbia', 'California', 'Canada', 'International', 'Oregon', 'PNW Regional', 'U.S. National', 'Washington']

functionalDomainPrimary
['Community Resilience', 'Earthquake Science', 'Emergency Management', 'Infrastructure Systems']

rolePrimary
['Coordination', 'Data Tools Provider', 'Emergency Response', 'Funding Provider', 'Infrastructure Operator', 'Knowledge Provider', 'Messaging Alerts Provider', 'Other', 'Policy Maker Regulator']

femaLifelinePrimary
['Communications', 'Energy', 'Food Hydration Shelter', 'Health Medical', 'Other', 'Safety Security', 'Transportation', 'Water Systems']



orgTypePrimary          geoPrimary      
Tribal Sovereign        Washington          30
Government              Oregon              29
                        U.S. National       19
                        Washington          18
Academic                U.S. National       16
Private Sector          U.S. National       13
Coordination Structure  U.S. National       11
Nonprofit Community     U.S. National       11
                        Oregon              11
Tribal Sovereign        California          11
Government              California          10
Tribal Sovereign        Oregon               9
Coordination Structure  PNW Regional         9
Academic                Oregon               6
Quasi Governmental      Oregon               6
                        Washington           5
Coordination Structure  Oregon               5
Government              British Columbia     4
Academic                California           4
Nonprofit Community     PNW Regional         4
dtype: int64

## notes and contact fields

In [35]:
TEXT_EXPORT_COLUMNS = [
    "name",
    "url",
    "key_contact",
    "contact_url",
    "summary",
    "review_flag",
    "review_note",
]

for col in TEXT_EXPORT_COLUMNS:
    df[col] = df[col].apply(normalize_text)

In [36]:
display(df[[
    "name",
    "summary",
    "review_flag",
    "review_note",
    "url",
    "key_contact",
    "contact_url",
]].head(10))

,name,summary,review_flag,review_note,url,key_contact,contact_url
0,211info,Provides community information and referral services that support disaster preparedness and recovery.,,,https://www.211info.org/,,
1,Alpine Fault Magnitude 8,,,,https://af8.org.nz/,,
2,Air National Guard,"Provides airlift, logistics, and emergency support capabilities for disaster response.",yes,Generic umbrella entry; consider replacing with state-specific Air National Guard entities.,https://www.ang.af.mil/,,
3,Global Security Systems/ALERT FM,Provides alerting technology and communications tools used for emergency warning and public information.,,,https://alertfm.com/,,
4,Alert Ready Emergency Alert System,Delivers public emergency alerts across Canada through participating broadcasters and wireless providers.,yes,Could be modeled as a national alerting system rather than a standalone program node.,https://www.alertready.ca/,,
5,Aon Impact Forecasting,,,,https://www.aon.com/en/capabilities/reinsurance/catastrophe-model-insight,,
6,American Society of Civil Engineers,Advances engineering practice and guidance relevant to resilient infrastructure and seismic risk reduction.,,,https://www.asce.org/,,
7,Affiliated Tribes of Northwest Indians,,,,https://atnitribes.org/,,
8,City of Bainbridge Island,"Provides local emergency preparedness, response, and recovery coordination.",,,https://www.bainbridgewa.gov/,,
9,Bathymetrix,"Bathymetrix is a geophysical consulting firm providing services in ocean acoustics, global seismology, and scientific data analysis.",,,https://bathymetrix.com/,,


In [37]:
def combine_notes(summary, review_flag, review_note):
    parts = []

    summary = normalize_text(summary)
    review_flag = normalize_text(review_flag)
    review_note = normalize_text(review_note)

    if summary:
        parts.append(summary)
    if review_flag:
        parts.append(f"Review flag: {review_flag}")
    if review_note:
        parts.append(f"Review note: {review_note}")

    return "\n\n".join(parts)

df["Organization Name"] = df["name"]
df["Org ID"] = df["node_id"]
df["Notes"] = df.apply(
    lambda row: combine_notes(row["summary"], row["review_flag"], row["review_note"]),
    axis=1,
)
df["Primary"] = df["key_contact"]
df["2ndry"] = df["contact_url"]

display(df[["Organization Name", "Org ID", "orgTypePrimary", "geoPrimary", "Notes", "Primary", "2ndry"]].head(10))

,Organization Name,Org ID,orgTypePrimary,geoPrimary,Notes,Primary,2ndry
0,211info,211info,Nonprofit Community,Oregon,Provides community information and referral services that support disaster preparedness and recovery.,,
1,Alpine Fault Magnitude 8,AF8,Coordination Structure,International,,,
2,Air National Guard,AirNG,Government,U.S. National,"Provides airlift, logistics, and emergency support capabilities for disaster response.\n\nReview flag: yes\n\nReview note: Generic umbrella entry; consider replacing with state-specific Air Nation...",,
3,Global Security Systems/ALERT FM,AlertFM,Private Sector,International,Provides alerting technology and communications tools used for emergency warning and public information.,,
4,Alert Ready Emergency Alert System,AlertReady,Coordination Structure,Canada,Delivers public emergency alerts across Canada through participating broadcasters and wireless providers.\n\nReview flag: yes\n\nReview note: Could be modeled as a national alerting system rather ...,,
5,Aon Impact Forecasting,AON,Private Sector,International,,,
6,American Society of Civil Engineers,ASCE,Nonprofit Community,U.S. National,Advances engineering practice and guidance relevant to resilient infrastructure and seismic risk reduction.,,
7,Affiliated Tribes of Northwest Indians,ATNI,Tribal Sovereign,PNW Regional,,,
8,City of Bainbridge Island,Bainbridge,Government,Washington,"Provides local emergency preparedness, response, and recovery coordination.",,
9,Bathymetrix,Bathymetrix,Private Sector,Oregon,"Bathymetrix is a geophysical consulting firm providing services in ocean acoustics, global seismology, and scientific data analysis.",,


In [38]:
df[[
    "Organization Name",
    "Org ID",
    "orgTypes",
    "orgTypePrimary",
    "geoTags",
    "geoPrimary",
    "nodeTypePrimary",
    "governanceLevelPrimary",
    "functionalDomainPrimary",
    "rolePrimary",
    "femaLifelinePrimary",
]].head(3)

,Organization Name,Org ID,orgTypes,orgTypePrimary,geoTags,geoPrimary,nodeTypePrimary,governanceLevelPrimary,functionalDomainPrimary,rolePrimary,femaLifelinePrimary
0,211info,211info,[Nonprofit Community],Nonprofit Community,[Oregon],Oregon,Organization,Non Governmental,Community Resilience,Coordination,Communications
1,Alpine Fault Magnitude 8,AF8,[Coordination Structure],Coordination Structure,[International],International,Hub,Multi,Community Resilience,Coordination,Other
2,Air National Guard,AirNG,[Government],Government,[U.S. National],U.S. National,Organization,State,Emergency Management,Infrastructure Operator,Safety Security


In [39]:
import json

df["Org ID"] = df["Org ID"].astype(str)

for col in [
    "orgTypes",
    "geoTags",
    "nodeTypes",
    "governanceLevels",
    "functionalDomains",
    "roleTags",
    "lifelineTags",
]:
    df[f"{col}_json"] = df[col].apply(json.dumps)

cols = [
    "Organization Name",
    "Org ID",
    "orgTypes_json",
    "orgTypePrimary",
    "geoPrimary",
    "Notes",
    "Primary",
    "2ndry",
    "geoTags_json",
    "nodeTypes_json",
    "nodeTypePrimary",
    "governanceLevels_json",
    "governanceLevelPrimary",
    "functionalDomains_json",
    "functionalDomainPrimary",
    "roleTags_json",
    "rolePrimary",
    "lifelineTags_json",
    "femaLifelinePrimary",
    "url",
    "review_flag",
    "review_note",
]

clean_df = df[cols].copy()

clean_df.to_csv("organizations_clean.csv", index=False)

print("Exported:", len(clean_df), "rows")
clean_df.head()

Exported: 284 rows


,Organization Name,Org ID,orgTypes_json,orgTypePrimary,geoPrimary,Notes,Primary,2ndry,geoTags_json,nodeTypes_json,...,governanceLevelPrimary,functionalDomains_json,functionalDomainPrimary,roleTags_json,rolePrimary,lifelineTags_json,femaLifelinePrimary,url,review_flag,review_note
0,211info,211info,"[""Nonprofit Community""]",Nonprofit Community,Oregon,Provides community information and referral services that support disaster preparedness and recovery.,,,"[""Oregon""]","[""Organization""]",...,Non Governmental,"[""Community Resilience""]",Community Resilience,"[""Coordination"", ""Data Tools Provider""]",Coordination,"[""Communications""]",Communications,https://www.211info.org/,,
1,Alpine Fault Magnitude 8,AF8,"[""Coordination Structure""]",Coordination Structure,International,,,,"[""International""]","[""Hub""]",...,Multi,"[""Community Resilience""]",Community Resilience,"[""Coordination"", ""Knowledge Provider"", ""Data Tools Provider""]",Coordination,"[""Other""]",Other,https://af8.org.nz/,,
2,Air National Guard,AirNG,"[""Government""]",Government,U.S. National,"Provides airlift, logistics, and emergency support capabilities for disaster response.\n\nReview flag: yes\n\nReview note: Generic umbrella entry; consider replacing with state-specific Air Nation...",,,"[""U.S. National""]","[""Organization""]",...,State,"[""Emergency Management""]",Emergency Management,"[""Infrastructure Operator""]",Infrastructure Operator,"[""Safety Security""]",Safety Security,https://www.ang.af.mil/,yes,Generic umbrella entry; consider replacing with state-specific Air National Guard entities.
3,Global Security Systems/ALERT FM,AlertFM,"[""Private Sector""]",Private Sector,International,Provides alerting technology and communications tools used for emergency warning and public information.,,,"[""International""]","[""Organization""]",...,Private,"[""Emergency Management""]",Emergency Management,"[""Messaging Alerts Provider"", ""Data Tools Provider""]",Messaging Alerts Provider,"[""Communications""]",Communications,https://alertfm.com/,,
4,Alert Ready Emergency Alert System,AlertReady,"[""Coordination Structure""]",Coordination Structure,Canada,Delivers public emergency alerts across Canada through participating broadcasters and wireless providers.\n\nReview flag: yes\n\nReview note: Could be modeled as a national alerting system rather ...,,,"[""Canada""]","[""Program""]",...,Multi,"[""Emergency Management""]",Emergency Management,"[""Messaging Alerts Provider""]",Messaging Alerts Provider,"[""Communications""]",Communications,https://www.alertready.ca/,yes,Could be modeled as a national alerting system rather than a standalone program node.


# looking at edges

In [40]:
df_edges = pd.read_excel(
    INFILE_EDGES,
    sheet_name="Relationships",
    engine="openpyxl"
)

print("Shape:", df_edges.shape)
df_edges.head()

Shape: (536, 5)


,From agency,To agency,Relationship type,Description,Status
0,BCHydro,NRCanGSC,data,NaN,NaN
1,BCHydro,NRCanGSC,tools/products,sharing models,NaN
2,NRCanGSC,BCHydro,data,NaN,NaN
3,NRCanGSC,BCHydro,tools/products,sharing models,NaN
4,CRESCENT,BCHydro,data,NaN,NaN


In [41]:
df_edges = df_edges.copy()

def clean_id(x):
    if pd.isna(x):
        return pd.NA
    # remove all whitespace, then (optional) strip punctuation the same way as nodes
    s = str(x).strip()
    s = pd.Series([s]).str.replace(r"\s+", "", regex=True).iloc[0]
    return s

df_edges["From agency"] = df_edges["From agency"].apply(clean_id)
df_edges["To agency"]   = df_edges["To agency"].apply(clean_id)

# Optional: also normalize Relationship type / Description / Status whitespace
for c in ["Relationship type", "Description", "Status"]:
    if c in df_edges.columns:
        df_edges[c] = (df_edges[c].astype("string")
                                   .str.strip()
                                   .str.replace(r"\s+", " ", regex=True))

# Drop *exact* duplicate rows (keeps first occurrence)
before = len(df_edges)
df_edges_clean = df_edges.drop_duplicates(keep="first").reset_index(drop=True)
after = len(df_edges_clean)

print(f"Rows before: {before}  |  after drop_duplicates: {after}  |  removed: {before-after}")
df_edges_clean.head()

Rows before: 536  |  after drop_duplicates: 520  |  removed: 16


,From agency,To agency,Relationship type,Description,Status
0,BCHydro,NRCanGSC,data,<NA>,<NA>
1,BCHydro,NRCanGSC,tools/products,sharing models,<NA>
2,NRCanGSC,BCHydro,data,<NA>,<NA>
3,NRCanGSC,BCHydro,tools/products,sharing models,<NA>
4,CRESCENT,BCHydro,data,<NA>,<NA>


In [42]:
df_edges_clean.to_csv("edges_clean.csv", index=False)
print("Wrote edges_clean.csv")

Wrote edges_clean.csv


# verify alignment

In [43]:
node_ids = set(clean_df["Org ID"].astype(str))

missing_from = sorted(set(df_edges_clean["From agency"]) - node_ids)
missing_to   = sorted(set(df_edges_clean["To agency"]) - node_ids)

print("edges FROM not in nodes:", missing_from)
print("----------------------")
print("edges TO not in nodes:", missing_to)

edges FROM not in nodes: ['ASF', 'BCAlert', 'BCEM', 'CAmedia', 'CaSSC', 'CalDWR', 'CalEAS', 'CalFire', 'CalTrans', 'ClackCo', 'Copes', 'DNR', 'EERi', 'IEMA', 'JPL', 'LidarBC', 'NHRP', 'NOAA', 'NOAA-NTWC', 'NOAA-NWS', 'NOAANWS', 'NOAANWTC', 'NRCanGSC', 'NTWC', 'NWS', 'ODHS', 'OR-Gov', 'ORDEQ', 'OREAS', 'OREM', 'ORmedia', 'PDXOEM', 'PMEL', 'PTWC', 'PWB', 'PacificCor', 'RCTWG', 'SeaGrant', 'Tribal', 'Trimet', 'UP', 'USGA', 'USGS', 'WADNRWGS', 'WAEAS', 'WAmedia', 'WPUC', 'copes', 'dogami', 'eeri', 'oem']
----------------------
edges TO not in nodes: ['BCEM', 'CAmedia', 'CaSSC', 'CalDWR', 'CalFire', 'CalTrans', 'ClackCo', 'Copes', 'DLCD', 'DNR', 'Earthscope', 'FIN', 'JPL', 'NHRP', 'NOAA', 'NOAA-NTWC', 'NOAA-NWS', 'NRCanGSC', 'NWS', 'ODHS', 'OR-Gov', 'ORDEQ', 'OREAS', 'OREM', 'ORmedia', 'Osspac', 'PDXOEM', 'PSC', 'PWB', 'RCTWG', 'SeaGrant', 'Sz4d', 'Tribal', 'Trimet', 'UP', 'USGS', 'WADNRWGS', 'WAmedia', 'copes', 'dogami', 'oem']


In [44]:
node_ids = set(clean_df["Org ID"].astype(str))

missing_from = sorted(node_ids - set(df_edges_clean["From agency"]))
missing_to   = sorted(node_ids - set(df_edges_clean["To agency"]))

print("Nodes not in edges From:", missing_from)
print("----------------------")

print("Nodes not in edges To", missing_to)

Nodes not in edges From: ['211info', 'AF8', 'AON', 'ASCE', 'ATNI', 'AirNG', 'AlertFM', 'AlertReady', 'BCEMCR', 'BCTransport', 'BLM', 'Bainbridge', 'Bathymetrix', 'BearRiver', 'BellPort', 'BigLagoon', 'BlueLakeRan', 'BoiseState', 'BurnsPaiute', 'CACC', 'CADWR', 'CALFIRE', 'CAParks', 'CBC', 'CERT', 'CHRN', 'CICOES', 'CLPUD', 'CLaSH', 'CMCA', 'COHORT', 'COStateU', 'CPBR', 'CPHumboldt', 'CRESA', 'CSM', 'CSSC', 'CTCLUSI', 'CUNA', 'CalAlerts', 'CalGuard', 'Caltrans', 'CascadiaCC', 'CdSPC', 'CedarLake', 'Chehalis', 'ClackamasEM', 'Colville', 'ConsejoHisp', 'CoosEM', 'Coquille', 'CowCreek', 'Cowlitz', 'EMCowichan', 'EPA', 'EPS', 'ESC', 'ESCERT', 'EVACNR', 'EWLabs', 'EarthScope', 'ElkValley', 'Eugene', 'Facet', 'FedMedia', 'FinCan', 'FraserBC', 'GHI', 'GLAD', 'GSC', 'GSOC', 'GrandRonde', 'HaleyAldr', 'Harvard', 'Hoh', 'HomeForward', 'Hoopa', 'IAEE', 'ICLR', 'IDStateU', 'IPREM', 'ImageCat', 'IndianaU', 'Jamestown', 'KMP', 'Kalispel', 'Karuk', 'KingCoEM', 'Klamath', 'KyotoU', 'L&Ccollege', 'LaneC

In [45]:
edge_endpoints = set(df_edges_clean["From agency"].dropna()) | set(df_edges_clean["To agency"].dropna())
org_ids = sorted(clean_df["Org ID"].dropna().astype(str).unique())

org_ids_with_edges = sorted([org_id for org_id in org_ids if org_id in edge_endpoints])
org_ids_without_edges = sorted([org_id for org_id in org_ids if org_id not in edge_endpoints])

summary_df = pd.DataFrame(
    {
        "status": ["has at least one edge", "has no edges"],
        "count": [len(org_ids_with_edges), len(org_ids_without_edges)],
        "share": [
            len(org_ids_with_edges) / len(org_ids) if org_ids else 0,
            len(org_ids_without_edges) / len(org_ids) if org_ids else 0,
        ],
    }
)

display(summary_df)

print("Org IDs with edges:")
print(org_ids_with_edges)
print()

print("Org IDs without edges")
print(org_ids_without_edges)
print()

display(
    clean_df.assign(has_edges=clean_df["Org ID"].astype(str).isin(edge_endpoints))
    .sort_values(["has_edges", "Org ID"], ascending=[False, True])
    [["Org ID", "Organization Name", "orgTypePrimary", "geoPrimary", "has_edges"]]
    .reset_index(drop=True)
)

,status,count,share
0,has at least one edge,53,0.18662
1,has no edges,231,0.81338


Org IDs with edges:
['BCHydro', 'BNSF', 'BPA', 'CGS', 'CLiP', 'CPUC', 'CRESCENT', 'CREW', 'CalOES', 'CoPesHub', 'DOGAMI', 'ECA', 'EEI', 'EERI', 'EWEB', 'FEMA', 'GEM', 'GLAD', 'GeoBC', 'Hakai', 'IAEE', 'MetroVan', 'NEHRP', 'NEMA', 'NHERI', 'NIFC', 'NTHMP', 'ODFW', 'ODOT', 'OEM', 'OHA', 'OMSI', 'OPRD', 'OPUC', 'ORMedia', 'OSSPAC', 'PACICC', 'PBEM', 'PBOT', 'PEER', 'PG&E', 'PGE', 'PNSN', 'PacifiCorp', 'RDPO', 'SCEC', 'SalemOEM', 'USACE', 'USBR', 'USFS', 'WAEMD', 'WEI', 'WSDOT']

Org IDs without edges
['211info', 'AF8', 'AON', 'ASCE', 'ATNI', 'AirNG', 'AlertFM', 'AlertReady', 'BCEMCR', 'BCTransport', 'BLM', 'Bainbridge', 'Bathymetrix', 'BearRiver', 'BellPort', 'BigLagoon', 'BlueLakeRan', 'BoiseState', 'BurnsPaiute', 'CACC', 'CADWR', 'CALFIRE', 'CAParks', 'CBC', 'CERT', 'CHRN', 'CICOES', 'CLPUD', 'CLaSH', 'CMCA', 'COHORT', 'COStateU', 'CPBR', 'CPHumboldt', 'CRESA', 'CSM', 'CSSC', 'CTCLUSI', 'CUNA', 'CalAlerts', 'CalGuard', 'Caltrans', 'CascadiaCC', 'CdSPC', 'CedarLake', 'Chehalis', 'Clackam

,Org ID,Organization Name,orgTypePrimary,geoPrimary,has_edges
0,BCHydro,BC Hydro,Quasi Governmental,British Columbia,True
1,BNSF,BNSF,Private Sector,U.S. National,True
2,BPA,Bonneville Power Administration,Quasi Governmental,PNW Regional,True
3,CGS,California Geological Survey,Government,California,True
4,CLiP,Cascadia Lifelines Program,Coordination Structure,Oregon,True
...,...,...,...,...,...
279,WarmSprings,Conferated Tribes of Warm Springs Oregon,Tribal Sovereign,Oregon,False
280,WashPost,Washington Post,Media,U.S. National,False
281,Wiyot,Wiyot Tribe,Tribal Sovereign,California,False
282,Yakama,Confederated Tribes and Bands of the Yakama Nation,Tribal Sovereign,Washington,False


In [47]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "scripts/preprocess_data.py"],
    capture_output=True,
    text=True,
)

print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)

result.check_returncode()
result


[preprocess-data] wrote public\data\graph.json (284 nodes, 233 edges)


CompletedProcess(args=['C:\\Users\\loicb\\anaconda3\\python.exe', 'scripts/preprocess_data.py'], returncode=0, stdout='[preprocess-data] wrote public\\data\\graph.json (284 nodes, 233 edges)\n', stderr='')